In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
import seaborn as sns
import sqlite3
import matplotlib as mpl
import matplotlib.pyplot as plt
import keras_tuner as kt

import copy
import random
import math
import sys
import gc

from sqlalchemy import create_engine
from tensorflow.keras import layers
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import CSVLogger
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import KFold
from sklearn import model_selection
from keras_tuner.engine.tuner import Tuner

from datetime import datetime

In [ ]:
sys.path.insert(0, "C:/PlayerProjectionsHelper")

In [ ]:
from FeatureEngineering import FeatureEngineering
from FeatureResampling import FeatureResampling
from Normalization import Normalization

In [ ]:
dbConnectionString = "sqlite:///baseball_info.db"
engine = create_engine(dbConnectionString)

In [ ]:
timeSeriesPitchingQuery = "SELECT SeasonStatsPitching.playerId,season,teams,Bios.dob,ipOuts,hits,singles,doubles,triples,homeRuns,runs,earnedRuns,walks,strikeOuts,hitByPitch,qualityStarts,averageFastballVelocity,zonePercentage,chasePercentage,swingingStrikePercentage,hardHitPercentage,barrelPercentage,groundBallPercentage,flyBallPercentage,lineDrivePercentage,popUpPercentage,strikeOutPercentage,walkPercentage,strikeOutWalkDifference,homeRunPerFlyBallPercentage,era,whip,ksPerNine FROM SeasonStatsPitching INNER JOIN Bios on Bios.playerId = SeasonStatsPitching.playerId WHERE season BETWEEN 2015 and 2025 and season != 2020 and SeasonStatsPitching.gamesStarted > 0 and saves = 0 and holds = 0 ORDER BY SeasonStatsPitching.playerId,season"

df = pd.read_sql(timeSeriesPitchingQuery, engine)

In [ ]:
len(df)

In [ ]:
subOneEraDf = df[df['era'] <= 1.5]

In [ ]:
len(subOneEraDf)

In [ ]:
df['qualityStarts'].std()

In [ ]:
df['qualityStarts'].head()

In [ ]:
#FEATURE ENGINEERING

In [ ]:
FeatureEngineering.dropPitchersWithNoBattersFaced(df)

In [ ]:
FeatureEngineering.dropOutlierEraPitchers(df)

In [ ]:
df = FeatureEngineering.dropPitchersWithOneSeasonOrLess(df)

In [ ]:
FeatureEngineering.replaceTeamsWithFirstTeamOfSeason(df)

In [ ]:
FeatureEngineering.replaceNullFastballVelocitiesWithMean(df)

In [ ]:
FeatureEngineering.replaceHomeRunPerFlyBallPercentageWithMaxValues(df)

In [ ]:
FeatureEngineering.replaceInfiniteEraAndWhipWithMaxValues(df)

In [ ]:
FeatureEngineering.getAgeFromUnixTimeStamp(df)

In [ ]:
featureNum = len(df.columns)
windowSize = 4

windowedFeatures, windowedLabels = FeatureEngineering.getWindowedFeaturesAndLabels(df, windowSize, featureNum)

In [ ]:
windowedLabels[0]

In [ ]:
windowedFeaturesArray = np.array(windowedFeatures)
windowedLabelsArray = np.array(windowedLabels)

In [ ]:
teamCategories = df['teams'].unique() 

windowedFeaturesArray = FeatureEngineering.oneHotEncodeTeamFeatures(windowedFeaturesArray, teamCategories)

In [ ]:
binnedLabelsArray = FeatureResampling.getBinnedLabelsArray(windowedLabelsArray)
sortedBinCount    = FeatureResampling.getSortedBinCountFromBinnedArray(binnedLabelsArray)

In [ ]:
labels = list(sortedBinCount.keys())
count  = list(sortedBinCount.values())

plt.figure(figsize=(6, 4))
bar = plt.bar(labels, count)

_ = plt.xticks(labels, rotation="vertical")
_ = plt.bar_label(bar, count)

In [ ]:
binIndexes = FeatureResampling.getConcatenatedBinIndexesFromLabels(labels)

In [ ]:
oneHotEncodedLength = len(teamCategories)

In [ ]:
stratifiedFeatures,stratifiedLabels = FeatureResampling.resampleFeaturesArrayForMinimumLabels(windowedFeaturesArray, windowedLabelsArray, binnedLabelsArray, binIndexes, 6, oneHotEncodedLength, 13)

In [ ]:
fullFeaturesArrayStratified = np.concatenate((windowedFeaturesArray, stratifiedFeatures), axis=0)
fullLabelsArrayStratified   = np.concatenate((windowedLabelsArray  , stratifiedLabels  ), axis=0)

In [ ]:
stratifiedBinnedLabelsArray = FeatureResampling.getBinnedLabelsArray(fullLabelsArrayStratified)
stratifiedSortedBinCount    = FeatureResampling.getSortedBinCountFromBinnedArray(stratifiedBinnedLabelsArray)

In [ ]:
stratifiedLabels = list(stratifiedSortedBinCount.keys())
stratifiedCount  = list(stratifiedSortedBinCount.values())

plt.figure(figsize=(6, 4))
bar = plt.bar(stratifiedLabels, stratifiedCount)

_ = plt.xticks(stratifiedLabels, rotation="vertical")
_ = plt.bar_label(bar, stratifiedCount)

In [ ]:
interpolatedFeaturesTest,interpolatedLabelsTest = FeatureResampling.applySmoteToFeaturesAndLabels(fullFeaturesArrayStratified, fullLabelsArrayStratified, 5, 500, oneHotEncodedLength, 13)

In [ ]:
interpolatedFeaturesTest = np.array(interpolatedFeaturesTest)
interpolatedFeaturesTest = interpolatedFeaturesTest.reshape(len(interpolatedFeaturesTest), 3, len(fullFeaturesArrayStratified[0][0]))
interpolatedLabelsTest   = np.array(interpolatedLabelsTest)

In [ ]:
interpolatedFeaturesTest = np.concatenate((interpolatedFeaturesTest, fullFeaturesArrayStratified), axis=0)
interpolatedLabelsTest   = np.concatenate((interpolatedLabelsTest  , fullLabelsArrayStratified), axis=0)

In [ ]:
iFeaturesNormalized,iLabelsNormalized,iFeaturesMax,iFeaturesMin,iLabelsMax,iLabelsMin = Normalization.normalizeTrainFeaturesAndLabels(interpolatedFeaturesTest, interpolatedLabelsTest)

In [ ]:
binnedSmoteLabelsArray = FeatureResampling.getBinnedLabelsArray(interpolatedLabelsTest)
sortedSmoteBinCount    = FeatureResampling.getSortedBinCountFromBinnedArray(binnedSmoteLabelsArray)

In [ ]:
smoteLabels = list(sortedSmoteBinCount.keys())
smoteCount  = list(sortedSmoteBinCount.values())

plt.figure(figsize=(6, 4))
bar = plt.bar(smoteLabels, smoteCount)

_ = plt.xticks(smoteLabels, rotation="vertical")
_ = plt.bar_label(bar, smoteCount)

In [ ]:
#TEST

In [ ]:
tunerEarlyStop = EarlyStopping(monitor='val_loss', mode='min', patience=10, restore_best_weights=True)

In [ ]:
#l2/l1 regularization, dropout, activation function, optmizier, loss function, hidden units, hidden layers, batch size, batch normalization
def getModelForTuner(hp):
    hp_learning_rate     = hp.Float('learning_rate'    , min_value=1e-7, max_value=1e-3, sampling='log')
    hp_weight_decay      = hp.Float('weight_decay'     , min_value=1e-7, max_value=1e-3, sampling='log')
    hp_huber_delta       = hp.Float('huber_delta'      , min_value=1.0 , max_value=10.0, sampling='log')

    num_rnn_layers = hp.Int("num_rnn_layers", min_value=1, max_value=2)
    
    model = tf.keras.Sequential()
    model.add(layers.Masking(mask_value=0.0, input_shape=(3, 60)))

    for i in range(num_rnn_layers):
        hp_units             = hp.Int  (f'units_{i}', min_value=5, max_value=30, step=6)
        hp_dropout           = hp.Float(f'dropout_{i}'          , min_value=0.1, max_value=0.9, sampling='linear')     
        hp_recurrent_dropout = hp.Float(f'recurrent_dropout_{i}', min_value=0.1, max_value=0.9, sampling='linear')  
        hp_regularizer       = hp.Float(f'regularizer_{i}'      , min_value=1e-7, max_value=1e-3, sampling='log')  

        if i < num_rnn_layers - 1:
            model.add(layers.Bidirectional(layers.LSTM(units=hp_units, dropout=hp_dropout, recurrent_dropout=hp_recurrent_dropout, kernel_regularizer=regularizers.l2(hp_regularizer), return_sequences=True)))
        else:
            model.add(layers.Bidirectional(layers.LSTM(units=hp_units, dropout=hp_dropout, recurrent_dropout=hp_recurrent_dropout, kernel_regularizer=regularizers.l2(hp_regularizer), return_sequences=False)))
        
        model.add(layers.LayerNormalization())
        
        model.add(layers.Dense(units = 4))

    model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=hp_learning_rate, weight_decay=hp_weight_decay), loss=tf.keras.losses.Huber(delta=hp_huber_delta), metrics=[tf.keras.metrics.R2Score(), "accuracy"])

    return model

In [ ]:
class KFoldTuner(Tuner):
    def run_trial(self, trial, x, y, *args, **kwargs):
        kFold = KFold(n_splits = 10, shuffle=True, random_state=42)

        epochs = trial.hyperparameters.get('tuner/epochs')

        batch_size = kwargs['batch_size']
        callbacks = kwargs['callbacks']

        valLossSum = 0.0

        for trainIndex,testIndex in kFold.split(x):
            tf.keras.backend.clear_session()
            tf.compat.v1.reset_default_graph()
            
            gc.collect()
            
            trainFeatures, valFeatures = x[trainIndex], x[testIndex]
            trainLabels  , valLabels   = y[trainIndex], y[testIndex]
    
            trainBinnedLabelsArrayT = FeatureResampling.getBinnedLabelsArray(trainLabels)
            trainSortedBinCountT    = FeatureResampling.getSortedBinCountFromBinnedArray(trainBinnedLabelsArrayT)

            trainLabelsBarT   = list(trainSortedBinCountT.keys())
            trainLabelsCountT = list(trainSortedBinCountT.values())

            trainBinIndexesT = FeatureResampling.getConcatenatedBinIndexesFromLabels(trainLabelsBarT)
    
            maxCountPerCluster = trainLabelsCountT[0]

            stratifiedTrainFeaturesResampled,stratifiedTrainLabelsResampled = FeatureResampling.resampleFeaturesArrayForMinimumLabels(trainFeatures, trainLabels, trainBinnedLabelsArrayT, trainBinIndexesT, 6, 30, 19)

            if len(stratifiedTrainFeaturesResampled) > 0:
                stratifiedTrainFeaturesResampled = np.concatenate((trainFeatures, stratifiedTrainFeaturesResampled), axis=0)
                stratifiedTrainLabelsResampled   = np.concatenate((trainLabels  , stratifiedTrainLabelsResampled  ), axis=0)
            else:
                stratifiedTrainFeaturesResampled = trainFeatures
                stratifiedTrainLabelsResampled = trainLabels
                
            interpolatedFeatures, interpolatedLabels = FeatureResampling.applySmoteToFeaturesAndLabels(stratifiedTrainFeaturesResampled, stratifiedTrainLabelsResampled, 5, maxCountPerCluster, 30, 19)
        
            interpolatedFeatures = np.array(interpolatedFeatures)
            interpolatedFeatures = interpolatedFeatures.reshape(len(interpolatedFeatures), 3, 60)
            interpolatedLabels   = np.array(interpolatedLabels)
        
            concatenatedFeaturesArray = np.concatenate((stratifiedTrainFeaturesResampled , interpolatedFeatures), axis=0)
            concatenatedLabelsArray   = np.concatenate((stratifiedTrainLabelsResampled   , interpolatedLabels  ), axis=0)
        
            normalizedFeatures,normalizedLabels,featuresMax,featuresMin,labelsMax,labelsMin  = Normalization.normalizeTrainFeaturesAndLabels(concatenatedFeaturesArray, concatenatedLabelsArray)
            normalizedValFeatures,normalizedValLabels                                          = Normalization.normalizeTestData(valFeatures, valLabels, featuresMax, featuresMin, labelsMax, labelsMin)
            
            model = self.hypermodel.build(trial.hyperparameters)

            model.fit(normalizedFeatures, normalizedLabels, batch_size=batch_size, epochs=epochs, validation_data=(normalizedValFeatures, normalizedValLabels), callbacks=callbacks)

            valLoss = model.evaluate(normalizedValFeatures,normalizedValLabels, verbose=0)[0]

            valLossSum += valLoss

            tf.keras.backend.clear_session()
            tf.compat.v1.reset_default_graph()
            
            gc.collect()

        valLossMean = valLossSum / 10.0

        print(f"val_loss mean: {valLossMean}")

        self.oracle.update_trial(trial.trial_id, {'val_loss': valLossMean})

In [ ]:
tuner = KFoldTuner(  
                     hypermodel=getModelForTuner,
                     oracle=kt.oracles.HyperbandOracle(
                         objective='val_loss',
                         max_epochs=100,
                         factor=3,
                         seed=42
                     )
                  )

In [ ]:
tuner.search(windowedFeaturesArray, windowedLabelsArray, batch_size=32, epochs=100, callbacks=[tunerEarlyStop])

In [ ]:
bestModelSoFar.summary()

In [ ]:
bestModelSoFar = tuner.hypermodel.build(tuner.get_best_hyperparameters(num_trials=1)[0])

In [ ]:
fullEarlyStop = EarlyStopping(monitor='loss', mode='min', patience=20, restore_best_weights=True)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='loss', # Metric to monitor
    factor=0.2,         # Factor by which the learning rate will be reduced
    patience=10,        # Number of epochs with no improvement after which learning rate will be reduced
    min_lr=0.00001      # Lower bound on the learning rate
)
    
bestModelSoFar.fit(normalizedFullFeatures, normalizedFullLabels, batch_size=64, epochs=250, callbacks=[fullEarlyStop, reduce_lr])

In [ ]:
# bestModel = tuner.get_best_models(num_models=1)[0]

# best_hps=tuner.get_best_hyperparameters(num_trials=1)[0]

# print(best_hps.get('num_rnn_layers'))
# print(best_hps.get('learning_rate'))
# print(best_hps.get('weight_decay'))
# print(best_hps.get('units_0'))
# print(best_hps.get('dropout_0'))
# print(best_hps.get('recurrent_dropout_0'))
# print(best_hps.get('regularizer_0'))

# print("-----------------")

# print(best_hps.get('units_1'))
# print(best_hps.get('dropout_1'))
# print(best_hps.get('recurrent_dropout_1'))
# print(best_hps.get('regularizer_1'))

# print("--------------------------")

# print(best_hps.get('units_2'))
# print(best_hps.get('dropout_2'))
# print(best_hps.get('recurrent_dropout_2'))
# print(best_hps.get('regularizer_2'))

In [ ]:
baselineResults = curModel.evaluate(iFeaturesNormalized, iLabelsNormalized)

In [ ]:
#END TEST

In [ ]:
def get_bidirectional_model():
    model = tf.keras.Sequential([
        layers.Masking(mask_value=0.0, input_shape=(3, 60)),
        #normalizeInput,
        layers.Bidirectional(layers.LSTM(units=11, dropout=0.6540530056254469, recurrent_dropout=0.7446258495961365, kernel_regularizer=regularizers.l2(1.0201163631160174e-06), return_sequences=True)),
        layers.LayerNormalization(),
        layers.Bidirectional(layers.LSTM(units=29, dropout=0.6992987457386866, recurrent_dropout=0.22041584308661513, kernel_regularizer=regularizers.l2(4.598913886300587e-07), return_sequences=True)),
        layers.LayerNormalization(),
        layers.Bidirectional(layers.LSTM(units=5, dropout=0.49625052752908416, recurrent_dropout=0.14394922575924696, kernel_regularizer=regularizers.l2(2.893733471157817e-05), return_sequences=False)),
        layers.LayerNormalization(),
        layers.Dense(units = 4)
    ])

    model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005652994819037546, weight_decay=8.504541632663274e-07), loss=tf.keras.losses.MeanSquaredError(), metrics=[tf.keras.metrics.R2Score(), "accuracy"])

    return model

In [ ]:
curModel = get_bidirectional_model()

curModel.summary()

In [ ]:
earlyStop = EarlyStopping(monitor='loss', mode='min', patience=20, restore_best_weights=True)

In [ ]:
bestModelSoFar.fit(iFeaturesNormalized, iLabelsNormalized, batch_size=32, epochs=500, callbacks=[earlyStop])

In [ ]:
flattenedInterpolatedFeatures = iFeaturesNormalized.reshape((-1, iFeaturesNormalized.shape[2]))        

In [ ]:
filteredInterpolatedFeatures = flattenedInterpolatedFeatures[:, 30:].copy()

In [ ]:
filteredInterpolatedFeatures[0]

In [ ]:
interpolatedFeaturesColumnNames = ['dob','ipOuts','hits','singles','doubles','triples','homeRuns','runs','earnedRuns','walks','strikeOuts','hitByPitch','qualityStarts','averageFastballVelocity','zonePercentage','chasePercentage','swingingStrikePercentage','hardHitPercentage','barrelPercentage','groundBallPercentage','flyBallPercentage','lineDrivePercentage','popUpPercentage','strikeOutPercentage','walkPercentage','strikeOutWalkDifference','homeRunPerFlyBallPercentage','era','whip', 'ksPerNine']

interpolatedFeaturesDf = pd.DataFrame(data=filteredInterpolatedFeatures,columns=interpolatedFeaturesColumnNames)

In [ ]:
interpolatedFeaturesDf.head()

In [ ]:
meltedDf = interpolatedFeaturesDf.melt(var_name='Column', value_name='Normalized')

In [ ]:
plt.figure(figsize=(12, 6))
ax = sns.violinplot(x='Column', y='Normalized', data=meltedDf)
_ = ax.set_xticklabels(interpolatedFeaturesDf.keys(), rotation=90)

In [ ]:
normalizedRegularData,normalizedRegularLabels,foo,bar,fizz,buzz = Normalization.normalizeTrainFeaturesAndLabels(windowedFeaturesArray, windowedLabelsArray)

In [ ]:
flattenedRegularFeatures = normalizedRegularData.reshape((-1, normalizedRegularData.shape[2]))
filteredNormalFeatures = flattenedRegularFeatures[:, 30:].copy()

normalFeaturesDf = pd.DataFrame(data=filteredNormalFeatures,columns=interpolatedFeaturesColumnNames)

In [ ]:
meltedNormalDf = normalFeaturesDf.melt(var_name='Column', value_name='Normalized')

In [ ]:
plt.figure(figsize=(12, 6))
ax = sns.violinplot(x='Column', y='Normalized', data=meltedNormalDf)
_ = ax.set_xticklabels(normalFeaturesDf.keys(), rotation=90)

In [ ]:
#iFeaturesMax,iFeaturesMin,iLabelsMax,iLabelsMin
indexToMax = {}

indexToMax[0] = iLabelsMax[0][0]
indexToMax[1] = iLabelsMax[0][1]
indexToMax[2] = iLabelsMax[0][2]
indexToMax[3] = iLabelsMax[0][3]

indexToMin = {}

indexToMin[0] = iLabelsMin[0][0]
indexToMin[1] = iLabelsMin[0][1]
indexToMin[2] = iLabelsMin[0][2]
indexToMin[3] = iLabelsMin[0][3]

season = 2025

hitterIds = []

hitterIdQuery = f"SELECT DISTINCT(SeasonStatsPitching.playerId),firstName,lastName from SeasonStatsPitching INNER JOIN Bios ON Bios.playerId = SeasonStatsPitching.playerId WHERE SeasonStatsPitching.season={season} and SeasonStatsPitching.gamesStarted > 0 and SeasonStatsPitching.saves = 0 and SeasonStatsPitching.holds = 0"

connection = sqlite3.connect("C:/baseball_info.db")
cursor     = connection.cursor()

cursor.execute(hitterIdQuery)

rows = cursor.fetchall()

for row in rows:
    hitterIds.append((row[0],row[1],row[2]))

cursor    .close()
connection.close()

print(len(hitterIds))

In [ ]:
playerPredictions = []

for hitter in hitterIds:
    playerId = hitter[0]

    playerFeaturesQuery = f"SELECT SeasonStatsPitching.playerId,season,teams,Bios.dob,ipOuts,hits,singles,doubles,triples,homeRuns,runs,earnedRuns,walks,strikeOuts,hitByPitch,qualityStarts,averageFastballVelocity,zonePercentage,chasePercentage,swingingStrikePercentage,hardHitPercentage,barrelPercentage,groundBallPercentage,flyBallPercentage,lineDrivePercentage,popUpPercentage,strikeOutPercentage,walkPercentage,strikeOutWalkDifference,homeRunPerFlyBallPercentage,era,whip,ksPerNine FROM SeasonStatsPitching INNER JOIN Bios on Bios.playerId = SeasonStatsPitching.playerId WHERE season BETWEEN 2023 and 2025 and SeasonStatsPitching.playerId =\"{playerId}\" ORDER BY SeasonStatsPitching.playerId,season"

    playerDf = pd.read_sql(playerFeaturesQuery, engine)

    FeatureEngineering.getAgeFromUnixTimeStamp(playerDf)
    FeatureEngineering.replaceTeamsWithFirstTeamOfSeason(playerDf)
    FeatureEngineering.replaceHomeRunPerFlyBallPercentageWithMaxValues(df)

    playerStats = []

    playerFrameArray = playerDf.values.tolist()
    
    for i in range(len(playerFrameArray)):
        playerStats.append(playerFrameArray[i][2:])
    
    if len(playerStats) < 3:
        diff = 3 - len(playerStats)
    
        while diff > 0:
            playerStats.append([0.0] * 31)
    
            diff -= 1

    playerWindow = np.array([playerStats])

    playerWindow = FeatureEngineering.oneHotEncodeTeamFeatures(playerWindow, teamCategories)

    featureMask = np.all(playerWindow == 0, axis=2)
    
    featureMask3d = np.dstack([featureMask] * playerWindow.shape[2])
    
    for i in range(len(featureMask3d)):
        for j in range(len(featureMask3d[i])):
            for k in range(len(featureMask3d[i][j])):
                if k < 30:
                    featureMask3d[i][j][k] = True
    
    maskedPlayerWindow = np.ma.masked_array(playerWindow, mask=featureMask3d)
    
    playerWindowMaskNormalized = (maskedPlayerWindow - iFeaturesMax) / (iFeaturesMin)

    playerWindowNormalized = playerWindowMaskNormalized.data

    playerPrediction = bestModelSoFar.predict(playerWindowNormalized)

    normalizedPlayerPrediction = (playerId, hitter[1], hitter[2])

    for i in range(len(playerPrediction[0])):
        prediction = playerPrediction[0][i]
    
        curMax  = iLabelsMax[0][i]
        curMin  = iLabelsMin[0][i]
    
        prediction = (prediction * curMin) + curMax
    
        if i == 0:
            prediction = max(0, math.ceil(prediction))
        else:
            prediction = max(0.0, round(prediction, 3))

        normalizedPlayerPrediction += (prediction,)

    playerPredictions.append(normalizedPlayerPrediction)

In [ ]:
maxQualityStarts = 0.0
minQualityStarts = 10000.0

maxEra = 0.0
minEra = 10000.0

maxWhip = 0.0
minWhip = 10000.0

maxKsPerNine = 0.0
minKsPerNine = 10000.0

for player in playerPredictions:
    qualityStarts = player[3]
    era           = player[4]
    whip          = player[5]
    ksPerNine     = player[6]

    maxQualityStarts = max(maxQualityStarts, qualityStarts)
    minQualityStarts = min(minQualityStarts, qualityStarts)

    maxEra = max(maxEra, era)
    minEra = min(minEra, era)

    maxWhip = max(maxWhip, whip)
    minWhip = min(minWhip, whip)

    maxKsPerNine = max(maxKsPerNine, ksPerNine)
    minKsPerNine = min(minKsPerNine, ksPerNine)

    
print(f"{maxQualityStarts}, {minQualityStarts}")
print(f"{maxEra}, {minEra}")
print(f"{maxWhip}, {minWhip}")
print(f"{maxKsPerNine}, {minKsPerNine}")

In [ ]:
def normalize(num, maxValue, minValue):
    if maxValue == 0 and minValue == 0:
        return 0
    if maxValue == minValue:
        return 0
    
    normalizedValue = (num - minValue) / (maxValue - minValue)

    return normalizedValue

In [ ]:
def reverse_normalization(num, maxValue, minValue):
    if maxValue == 0 and minValue == 0:
        return 0
    if maxValue == minValue:
        return 0
    
    reverseNormalizedValue = 1 - ((num - minValue) / (maxValue - minValue))

    return reverseNormalizedValue

In [ ]:
for i in range(len(playerPredictions)):
    player = playerPredictions[i]
    
    qualityStarts = player[3]
    era           = player[4]
    whip          = player[5]
    ksPerNine     = player[6]

    normalizedQualityStarts = normalize            (qualityStarts, maxQualityStarts, minQualityStarts)
    normalizedEra           = reverse_normalization(era, maxEra, minEra)
    normalizedWhip          = reverse_normalization(whip, maxWhip, minWhip)
    normalizedKsPerNine     = normalize            (ksPerNine, maxKsPerNine, minKsPerNine)

    overallGrade = normalizedEra + normalizedWhip + normalizedKsPerNine

    player += (overallGrade,)

    playerPredictions[i] = player

In [ ]:
gradeSortedPredictions = sorted(playerPredictions, key=lambda player: player[7], reverse=True)

In [ ]:
for i in range(100):
    print(gradeSortedPredictions[i])

In [ ]:
for featureIndex in range(len(teamCategories), iFeaturesNormalized.shape[2]):
   shuffledTestFeatures = iFeaturesNormalized.copy()

   testColumnElements = [shuffledTestFeatures[i][j][featureIndex] for i in range(len(shuffledTestFeatures)) for j in range(len(shuffledTestFeatures[i]))]

   random.shuffle(testColumnElements)
   
   k = 0
   
   for i in range(len(shuffledTestFeatures)):
       for j in range(len(shuffledTestFeatures[i])):
           shuffledTestFeatures[i][j][featureIndex] = testColumnElements[k]
   
           k += 1

   shuffledResults = curModel.evaluate(shuffledTestFeatures, iLabelsNormalized, verbose=0)

   ratio = shuffledResults[0] / baselineResults[0]

   print(f"{df.columns[featureIndex+3-len(teamCategories)]}: {ratio}, {shuffledResults[0]}")

In [ ]:
print(baselineResults[0])

In [ ]:
predictionHeaders = ["id", "firstName", "lastName", "qualityStarts", "era", "whip", "ksPerNine", "grade"]

filePath = "./starting_pitcher_predictions_2026.csv"

predictionsDf = pd.DataFrame(gradeSortedPredictions, columns=predictionHeaders)

predictionsDf.to_csv(filePath, index=False)

In [ ]:
bestModelSoFar.save("C:/best_3_9_2026/starting_pitcher_projection_model.keras")